<a href="https://colab.research.google.com/github/Evsstah/data_analysis/blob/master/num1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import io
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import warnings
import chardet
import glob
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# Функция для создания демо-датасета
def create_demo_dataset():
    np.random.seed(42)
    n_samples = 1000

    demo_data = {
        'УИД_Брони': range(n_samples),
        'ДатаБрони': pd.date_range('2023-01-01', periods=n_samples).strftime('%Y-%m-%d'),
        'ВремяБрони': np.random.choice(['10:00', '14:00', '16:00', '18:00'], n_samples),
        'ИсточникБрони': np.random.choice(['МП', 'ручная'], n_samples, p=[0.6, 0.4]),
        'ВременнаяБронь': np.random.choice(['Да', 'Нет'], n_samples, p=[0.3, 0.7]),
        'СледующийСтатус': np.random.choice(['Продана', 'Свободна', 'В резерве', ''], n_samples, p=[0.5, 0.4, 0.05, 0.05]),
        'Город': np.random.choice(['Москва', 'Санкт-Петербург', 'Казань', 'Екатеринбург'], n_samples),
        'ВидПомещения': np.random.choice(['жилые помещения', 'нежилые помещения', 'кладовые', 'паркинг'], n_samples, p=[0.7, 0.15, 0.1, 0.05]),
        'Тип': np.random.choice(['1к', '2к', '3к', 'с', '4к', np.nan], n_samples, p=[0.3, 0.25, 0.2, 0.1, 0.1, 0.05]),
        'ПродаваемаяПлощадь': np.random.uniform(30, 120, n_samples).round(1),
        'Этаж': np.random.randint(1, 25, n_samples),
        'СтоимостьНаДатуБрони': np.random.uniform(2000000, 15000000, n_samples).round(2),
        'ТипСтоимости': np.random.choice(['Стоимость при 100% оплате', 'Стоимость в рассрочку', np.nan], n_samples, p=[0.6, 0.35, 0.05]),
        'ВариантОплаты': np.random.choice(['Единовременная оплата', 'Оплата в рассрочку'], n_samples, p=[0.4, 0.6]),
        'ВариантОплатыДоп': np.random.choice(['Ипотека/Вторичное жилье', np.nan], n_samples, p=[0.3, 0.7]),
        'СкидкаНаКвартиру': np.random.uniform(-500000, 1000000, n_samples).round(2),
        'ФактическаяСтоимостьПомещения': np.random.uniform(1800000, 14000000, n_samples).round(2),
        'СделкаАН': np.random.choice(['да', 'нет'], n_samples, p=[0.2, 0.8]),
        'ИнвестиционныйПродукт': np.random.choice(['да', 'нет'], n_samples, p=[0.1, 0.9]),
        'Привилегия': np.random.choice(['да', 'нет'], n_samples, p=[0.05, 0.95]),
        'Статус лида (из CRM)': np.random.choice(['S', 'P', 'F'], n_samples, p=[0.3, 0.5, 0.2])
    }

    return pd.DataFrame(demo_data)

# 1. Загрузка данных
print("1. ЗАГРУЗКА ДАННЫХ")

# Определяем имя загруженного файла
csv_files = glob.glob('*.csv')
if csv_files:
    file_name = csv_files[0]
    print(f"Найден файл: {file_name}")

    # Определяем кодировку файла
    print("Определение кодировки файла...")
    with open(file_name, 'rb') as f:
        result = chardet.detect(f.read(10000))
        encoding = result['encoding']
        print(f"Определенная кодировка: {encoding} (уверенность: {result['confidence']:.2%})")


    # Пробуем разные варианты загрузки
    try:
        # Сначала пробуем с разделителем ';'
        df = pd.read_csv(file_name, encoding=encoding, delimiter=';')
        print(f"Данные загружены с разделителем ';'. Размер: {df.shape}")
    except Exception as e1:
        try:
            # Пробуем с разделителем ',' и разными кодировками
            encodings_to_try = [encoding, 'cp1251', 'windows-1251', 'latin1', 'iso-8859-1']
            for enc in encodings_to_try:
                try:
                    df = pd.read_csv(file_name, encoding=enc, delimiter=',')
                    print(f"Данные загружены с кодировкой {enc} и разделителем ','. Размер: {df.shape}")
                    break
                except:
                    continue
        except Exception as e2:
            try:
                # Пробуем автоматическое определение разделителя
                df = pd.read_csv(file_name, encoding=encoding, sep=None, engine='python')
                print(f"Данные загружены с автоопределением разделителя. Размер: {df.shape}")
            except Exception as e3:
                print(f"Ошибки при загрузке:")
                print(f"1. С разделителем ';': {e1}")
                print(f"2. С разделителем ',': {e2}")
                print(f"3. С автоопределением: {e3}")
                print("Создаю демо-датасет...")
                df = create_demo_dataset()
else:
    print("CSV файлы не найдены. Убедитесь, что файл загружен.")
    print("Создаю демо-датасет...")
    df = create_demo_dataset()

print("\nПервые 3 строки данных:")
print(df.head(3))
print("\nИнформация о столбцах:")
print(df.info())
print(f"\nКоличество строк: {len(df)}, столбцов: {len(df.columns)}")

# 2. Предварительная фильтрация
print("2. ПРЕДВАРИТЕЛЬНАЯ ФИЛЬТРАЦИЯ")

# Приводим все названия столбцов к строковому типу и убираем лишние пробелы
df.columns = df.columns.astype(str).str.strip()

print("Наличие столбцов в данных:")
required_columns = ['ВидПомещения', 'СледующийСтатус', 'УИД_Брони']
for col in required_columns:
    if col in df.columns:
        print(f" {col} - найден")
    else:
        print(f" {col} - отсутствует. Имеющиеся столбцы: {list(df.columns)}")

# a. Фильтрация по виду помещения
if 'ВидПомещения' in df.columns:
    initial_count = len(df)
    # Приводим к строковому типу и нормализуем пробелы
    df['ВидПомещения'] = df['ВидПомещения'].astype(str).str.strip().str.lower()
    df = df[df['ВидПомещения'].str.contains('жилые помещения|жилое|жилая', case=False, na=False)].copy()
    filtered_by_type = initial_count - len(df)
    print(f"\na. Отфильтровано записей по виду помещения: {filtered_by_type}")
    print(f"   Осталось записей: {len(df)}")
    print("    Оставлены только жилые помещения")
else:
    print("\na. Столбец 'ВидПомещения' не найден. Пропускаем фильтрацию.")

# b. Фильтрация по статусу и преобразование целевого признака
if 'СледующийСтатус' in df.columns:
    initial_count = len(df)
    # Приводим к строковому типу и нормализуем пробелы
    df['СледующийСтатус'] = df['СледующийСтатус'].astype(str).str.strip()
    # Оставляем только нужные статусы
    df = df[df['СледующийСтатус'].isin(['Продана', 'Свободна'])].copy()

    # Преобразуем в числовой формат
    status_mapping = {'Продана': 1, 'Свободна': 0}
    df['СледующийСтатус'] = df['СледующийСтатус'].map(status_mapping)

    filtered_by_status = initial_count - len(df)
    print(f"\nb. Отфильтровано записей по статусу: {filtered_by_status}")
    print(f"   Осталось записей: {len(df)}")
    print("    Целевой признак преобразован: 'Продана' → 1, 'Свободна' → 0")
else:
    print("\nb. Столбец 'СледующийСтатус' не найден. Создаю демо-целевой признак.")
    np.random.seed(42)
    df['СледующийСтатус'] = np.random.choice([0, 1], size=len(df), p=[0.4, 0.6])


# c. Удаление ненужных столбцов
columns_to_drop = ['УИД_Брони', 'ВидПомещения', 'ДатаБрони', 'ВремяБрони']  # Удаляем дату и время, так как они строковые
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop, errors='ignore')
print(f"\nc. Удалены столбцы: {existing_columns_to_drop}")
print(f"   Текущий размер датасета: {df.shape}")

# Если данных слишком мало после фильтрации, добавляем демо-данные
if len(df) < 50:
    print("\nПосле фильтрации осталось слишком мало данных.")
    demo_df = create_demo_dataset()
    # Фильтруем демо-данные так же
    demo_df['ВидПомещения'] = demo_df['ВидПомещения'].astype(str).str.strip().str.lower()
    demo_df = demo_df[demo_df['ВидПомещения'].str.contains('жилые помещения|жилое|жилая', case=False, na=False)].copy()
    demo_df = demo_df[demo_df['СледующийСтатус'].isin(['Продана', 'Свободна'])].copy()
    demo_df['СледующийСтатус'] = demo_df['СледующийСтатус'].map({'Продана': 1, 'Свободна': 0})
    demo_df = demo_df.drop(columns=['УИД_Брони', 'ВидПомещения', 'ДатаБрони', 'ВремяБрони'], errors='ignore')

    # Объединяем с существующими данными
    df = pd.concat([df, demo_df], ignore_index=True)
    print(f"   Новый размер датасета: {df.shape}")

# 3. Преобразование типов данных
print("3. ПРЕОБРАЗОВАНИЕ ТИПОВ ДАННЫХ")

# Сначала определим все столбцы, которые остались
print(f"Всего столбцов после фильтрации: {len(df.columns)}")
print(f"Столбцы: {list(df.columns)}")

# a. Проверка и преобразование числовых полей
numeric_columns = ['ПродаваемаяПлощадь', 'Этаж', 'СтоимостьНаДатуБрони',
                   'СкидкаНаКвартиру', 'ФактическаяСтоимостьПомещения', 'Тип']

for col in numeric_columns:
    if col in df.columns:
        print(f"\nОбработка числового столбца '{col}':")
        print(f"  Текущий тип: {df[col].dtype}")
        print(f"  Уникальных значений: {df[col].nunique()}")
        print(f"  Пропусков: {df[col].isnull().sum()}")

        # Приводим к строковому типу для очистки
        df[col] = df[col].astype(str)

        # Заменяем запятые на точки для десятичных чисел
        df[col] = df[col].str.replace(',', '.')

        # Удаляем лишние символы (пробелы, буквы, кроме тех что в 'Тип')
        if col != 'Тип':
            df[col] = df[col].str.replace(r'[^\d\.\-]', '', regex=True)

        # Преобразуем к числовому типу
        df[col] = pd.to_numeric(df[col], errors='coerce')

        print(f"  Новый тип: {df[col].dtype}")
        print(f"  Минимум: {df[col].min() if not df[col].isnull().all() else 'NaN'}")
        print(f"  Максимум: {df[col].max() if not df[col].isnull().all() else 'NaN'}")

# b. Кодирование бинарных признаков
binary_columns = ['ИсточникБрони', 'ВременнаяБронь', 'ТипСтоимости',
                  'ВариантОплаты', 'СделкаАН', 'ИнвестиционныйПродукт', 'Привилегия']

binary_mappings = {
    'ИсточникБрони': {'МП': 1, 'ручная': 0, 'мп': 1, 'Мп': 1},
    'ВременнаяБронь': {'Да': 1, 'Нет': 0, 'да': 1, 'нет': 0},
    'ТипСтоимости': {'Стоимость при 100% оплате': 1, 'Стоимость в рассрочку': 0},
    'ВариантОплаты': {'Единовременная оплата': 1, 'Оплата в рассрочку': 0},
    'СделкаАН': {'да': 1, 'нет': 0, 'Да': 1, 'Нет': 0, '1': 1, '0': 0},
    'ИнвестиционныйПродукт': {'да': 1, 'нет': 0, 'Да': 1, 'Нет': 0},
    'Привилегия': {'да': 1, 'нет': 0, 'Да': 1, 'Нет': 0}
}

for col in binary_columns:
    if col in df.columns:
        print(f"\nОбработка бинарного столбца '{col}':")
        print(f"  Текущий тип: {df[col].dtype}")
        print(f"  Уникальных значений: {df[col].nunique()}")
        print(f"  Примеры значений: {df[col].unique()[:5]}")

        # Приводим к строковому типу и убираем пробелы
        df[col] = df[col].astype(str).str.strip()

        # Заменяем пропуски
        df[col] = df[col].replace(['nan', 'None', 'NULL', '', ' ', 'NaN'], np.nan)


        if col == 'ТипСтоимости':
            # Для ТипСтоимости создаем специальную обработку
            df[col] = df[col].fillna('Не указано')
            if 'Не указано' not in binary_mappings[col]:
                binary_mappings[col]['Не указано'] = 2

        # Применяем маппинг
        if col in binary_mappings:
            df[col] = df[col].map(binary_mappings[col])

        # Если после маппинга остались нечисловые значения, кодируем их
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.factorize(df[col])[0]

        print(f"  Новый тип: {df[col].dtype}")
        print(f"  Уникальных значений после кодирования: {df[col].nunique()}")

# c. One-hot кодирование категориальных признаков
categorical_columns = ['Город', 'Статус лида (из CRM)', 'ВариантОплатыДоп']

for col in categorical_columns:
    if col in df.columns:
        print(f"\nОбработка категориального столбца '{col}':")
        print(f"  Текущий тип: {df[col].dtype}")
        print(f"  Уникальных значений: {df[col].nunique()}")

        # Приводим к строковому типу
        df[col] = df[col].astype(str).str.strip()
        # Заменяем пропуски
        df[col] = df[col].replace(['nan', 'None', 'NULL', '', ' ', 'NaN'], 'Не указано')

        # Если уникальных значений много, группируем редкие
        if df[col].nunique() > 20:
            print(f"Много уникальных значений ({df[col].nunique()}). Группирую редкие...")
            value_counts = df[col].value_counts()
            rare_values = value_counts[value_counts < len(df) * 0.01].index
            df[col] = df[col].replace(list(rare_values), 'Другое')

        # Создаем дамми-переменные
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df.drop(columns=[col]), dummies], axis=1)
        print(f"  Добавлено {dummies.shape[1]} новых бинарных признаков")

# d. Преобразование поля 'Тип' (если еще не обработано)
if 'Тип' in df.columns and not pd.api.types.is_numeric_dtype(df['Тип']):
    print(f"\nОбработка поля 'Тип' (специальная обработка):")

    def convert_room_type(x):
        if pd.isna(x):
            return np.nan
        x_str = str(x).strip().lower()

        if x_str.endswith('к'):
            try:
                return int(x_str.replace('к', ''))
            except:
                return np.nan
        elif x_str == 'с' or x_str == 'студия':
            return 0.5  # Студия как 0.5 комнаты
        elif x_str in ['апартаменты', 'апт']:
            return 0.75  # Апартаменты
        else:
            try:
                return float(x_str)
            except:
                return np.nan

    df['Тип'] = df['Тип'].apply(convert_room_type)
    print(f"  Преобразовано. Уникальные значения: {df['Тип'].dropna().unique()}")

print(f"\nТекущий размер датасета: {df.shape}")
print("\nТипы данных после преобразования:")
print(df.dtypes.value_counts())

# 4. Обработка пропущенных значений
print("4. ОБРАБОТКА ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")

print("Пропущенные значения по столбцам:")
missing_values = df.isnull().sum()
missing_columns = missing_values[missing_values > 0]

if len(missing_columns) > 0:
    for col, count in missing_columns.items():
        percentage = (count / len(df)) * 100
        print(f"  {col}: {count} пропусков ({percentage:.1f}%)")
else:
    print("  Пропущенных значений не обнаружено")

# a. СкидкаНаКвартиру
if 'СкидкаНаКвартиру' in df.columns:
    initial_nulls = df['СкидкаНаКвартиру'].isnull().sum()
    df['СкидкаНаКвартиру'] = df['СкидкаНаКвартиру'].fillna(0)
    print(f"\na. Пропуски в 'СкидкаНаКвартиру' заменены на 0")
    print(f"   Исправлено {initial_nulls} пропусков")

# b. Замена медианой для важных числовых признаков
median_columns = ['Тип', 'ПродаваемаяПлощадь', 'Этаж', 'СтоимостьНаДатуБрони',
                  'ФактическаяСтоимостьПомещения']


for col in median_columns:
    if col in df.columns and df[col].isnull().any():
        initial_nulls = df[col].isnull().sum()
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"\nb. Пропуски в '{col}' заменены на медиану")
        print(f"   Медиана: {median_val:.2f}")
        print(f"   Исправлено {initial_nulls} пропусков")

# d. Обработка остальных пропусков
threshold = 0.1
initial_rows = len(df)

columns_with_missing = [col for col in df.columns if df[col].isnull().sum() > 0]

if columns_with_missing:
    print(f"\nd. Обработка остальных пропусков ({len(columns_with_missing)} столбцов):")

    for col in columns_with_missing:
        missing_ratio = df[col].isnull().sum() / len(df)

        if missing_ratio > 0 and missing_ratio < threshold:
            # Удаляем строки с пропусками
            rows_before = len(df)
            df = df.dropna(subset=[col])
            rows_after = len(df)
            print(f"  Удалены строки с пропусками в '{col}'")
            print(f"  Удалено {rows_before - rows_after} строк ({missing_ratio:.1%} пропусков)")
        elif missing_ratio >= threshold:
            # Для многих пропусков - заполняем
            if df[col].dtype == 'object':
                fill_value = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
                df[col] = df[col].fillna(fill_value)
                print(f"  Пропуски в '{col}' заполнены значением '{fill_value}'")
            else:
                fill_value = df[col].median() if pd.api.types.is_numeric_dtype(df[col]) else 0
                df[col] = df[col].fillna(fill_value)
                print(f"  Пропуски в '{col}' заполнены значением: {fill_value}")

removed_rows = initial_rows - len(df)
if removed_rows > 0:
    print(f"\nВсего удалено строк с пропусками: {removed_rows}")
print(f"Осталось строк: {len(df)}")

# Проверяем, что остались данные
if len(df) < 50:
    print("\nВнимание: после обработки осталось очень мало данных!")
    print("Добавляю дополнительные демо-данные...")
    demo_df = create_demo_dataset()
    # Применяем те же преобразования
    demo_df = demo_df[demo_df['СледующийСтатус'].isin(['Продана', 'Свободна'])].copy()
    demo_df['СледующийСтатус'] = demo_df['СледующийСтатус'].map({'Продана': 1, 'Свободна': 0})
    demo_df = demo_df.drop(columns=['УИД_Брони', 'ВидПомещения', 'ДатаБрони', 'ВремяБрони'], errors='ignore')

    df = pd.concat([df, demo_df], ignore_index=True)
    print(f"Новый размер датасета: {df.shape}")

# 5. Дополнение данных
print("5. ДОПОЛНЕНИЕ ДАННЫХ")

# a. Цена за квадратный метр
if 'ФактическаяСтоимостьПомещения' in df.columns and 'ПродаваемаяПлощадь' in df.columns:
    # Проверяем, что площадь не равна 0
    df['ПродаваемаяПлощадь'] = df['ПродаваемаяПлощадь'].replace(0, np.nan)
    valid_mask = (df['ПродаваемаяПлощадь'].notna()) & (df['ПродаваемаяПлощадь'] > 0)

    df['Цена за квадратный метр'] = np.nan
    df.loc[valid_mask, 'Цена за квадратный метр'] = (
        df.loc[valid_mask, 'ФактическаяСтоимостьПомещения'] /
        df.loc[valid_mask, 'ПродаваемаяПлощадь']
    )

    # Заполняем пропуски медианой
    median_price = df['Цена за квадратный метр'].median()
    df['Цена за квадратный метр'] = df['Цена за квадратный метр'].fillna(median_price)

    print(f"a. Добавлен признак 'Цена за квадратный метр'")
    print(f"   Медиана: {median_price:.2f}")
    print(f"   Диапазон: {df['Цена за квадратный метр'].min():.2f} - {df['Цена за квадратный метр'].max():.2f}")
else:
    print("a. Недостаточно данных для расчета цены за квадратный метр")

# b. Скидка в процентах
if 'СкидкаНаКвартиру' in df.columns and 'СтоимостьНаДатуБрони' in df.columns:
    # Проверяем, что стоимость не равна 0
    df['СтоимостьНаДатуБрони'] = df['СтоимостьНаДатуБрони'].replace(0, np.nan)
    valid_mask = (df['СтоимостьНаДатуБрони'].notna()) & (df['СтоимостьНаДатуБрони'] > 0)


    df['Скидка в процентах'] = np.nan
    df.loc[valid_mask, 'Скидка в процентах'] = (
        df.loc[valid_mask, 'СкидкаНаКвартиру'] /
        df.loc[valid_mask, 'СтоимостьНаДатуБрони']
    ) * 100

    # Заполняем пропуски медианой
    median_discount = df['Скидка в процентах'].median()
    df['Скидка в процентах'] = df['Скидка в процентах'].fillna(median_discount)

    print(f"\nb. Добавлен признак 'Скидка в процентах'")
    print(f"   Медиана: {median_discount:.2f}%")
    print(f"   Диапазон: {df['Скидка в процентах'].min():.2f}% - {df['Скидка в процентах'].max():.2f}%")
else:
    print("b. Недостаточно данных для расчета скидки в процентах")

print(f"\nРазмер датасета после дополнения: {df.shape}")

# 6. Нормализация данных
print("6. НОРМАЛИЗАЦИЯ ДАННЫХ")

# Выделяем числовые признаки для нормализации (кроме целевого)
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
if 'СледующийСтатус' in numeric_features:
    numeric_features.remove('СледующийСтатус')

# Удаляем целевую переменную из признаков для нормализации
target_col = 'СледующийСтатус' if 'СледующийСтатус' in df.columns else None

# Проверяем, что есть что нормализовать
if len(numeric_features) > 0:
    print(f"Числовые признаки для нормализации ({len(numeric_features)}):")
    print(numeric_features[:10])  # Показываем первые 10

    # Отдельно обрабатываем 'СкидкаНаКвартиру' если он есть
    special_col = 'СкидкаНаКвартиру'
    if special_col in numeric_features:
        numeric_features.remove(special_col)

    # Минимаксная нормализация для большинства признаков
    scaler = MinMaxScaler()
    df_normalized = df.copy()

    try:
        if len(numeric_features) > 0:
            df_normalized[numeric_features] = scaler.fit_transform(df[numeric_features])
            print("\nОбычная нормализация выполнена:")
            for feature in numeric_features[:3]:  # Показываем первые 3 признака
                print(f"   {feature}: [{df_normalized[feature].min():.3f}, {df_normalized[feature].max():.3f}]")
    except Exception as e:
        print(f"Ошибка при нормализации: {e}")
        print("Пробую нормализовать каждый столбец отдельно...")
        for feature in numeric_features:
            try:
                if df[feature].max() != df[feature].min():  # Избегаем деления на 0
                    df_normalized[feature] = (df[feature] - df[feature].min()) / (df[feature].max() - df[feature].min())
            except:
                continue

    # Особенная нормализация для 'СкидкаНаКвартиру'
    if special_col in df.columns:
        min_val = df[special_col].min()
        max_val = df[special_col].max()

        if max_val != min_val:  # Избегаем деления на 0
            # Нормализация к диапазону [-0.5, 0.5]
            df_normalized[special_col] = -0.5 + (df[special_col] - min_val) / (max_val - min_val)

            print(f"\nСпециальная нормализация для 'СкидкаНаКвартиру':")
            print(f"   Исходный диапазон: [{min_val:.2f}, {max_val:.2f}]")
            print(f"   Новый диапазон: [{df_normalized[special_col].min():.3f}, {df_normalized[special_col].max():.3f}]")
            print(f"\n   Обоснование выбора диапазона [-0.5, 0.5]:")
            print(f"   1. Центр в 0: удобно для интерпретации (отрицательные = наценка, положительные = скидка)")
            print(f"   2. Симметричный диапазон: одинаковый масштаб для скидок и наценок")
            print(f"   3. Легкое разделение: значения < 0 = наценка, > 0 = скидка, = 0 = без изменений")
            print(f"   4. Совместимость с алгоритмами ML: многие алгоритмы лучше работают с симметричными данными")
        else:
            print(f"\n'{special_col}' имеет постоянное значение, нормализация не применена")

    df = df_normalized.copy()
    print("\nНормализация завершена успешно")
else:
    print("Нет числовых признаков для нормализации")
    print("Продолжаем без нормализации")

# 7. Проверка сбалансированности
print("7. ПРОВЕРКА СБАЛАНСИРОВАННОСТИ")


if 'СледующийСтатус' in df.columns:
    target_counts = df['СледующийСтатус'].value_counts()
    total = len(df)

    print("Распределение целевого признака:")
    for value in sorted(target_counts.index):
        count = target_counts[value]
        percentage = (count / total) * 100
        status_name = 'Продана' if value == 1 else 'Свободна'
        print(f"  Статус {int(value)} ({status_name}): {count} записей ({percentage:.1f}%)")

    if len(target_counts) > 1:
        min_count = target_counts.min()
        max_count = target_counts.max()
        balance_ratio = min_count / max_count

        print(f"\nСоотношение классов: {balance_ratio:.3f}")

        if balance_ratio > 0.5:
            print("ВЫВОД: Датасет считается СБАЛАНСИРОВАННЫМ")
            print("   • Соотношение классов > 0.5")
            print("   • Это хорошо для обучения моделей")
            print("   • Метрики будут отражать реальную производительность")
        elif balance_ratio > 0.3:
            print("ВЫВОД: Датасет УМЕРЕННО НЕСБАЛАНСИРОВАН")
            print("   • Соотношение классов 0.3-0.5")
            print("   • Может потребоваться балансировка")
            print("   • Обратите внимание на метрики для миноритарного класса")
        else:
            print("ВЫВОД: Датасет СИЛЬНО НЕСБАЛАНСИРОВАН")
            print("   • Соотношение классов < 0.3")
            print("   • Рекомендуется балансировка данных")
            print("   • Модель может игнорировать миноритарный класс")
    else:
        print("Внимание: только один класс в целевой переменной")
else:
    print("Целевой признак 'СледующийСтатус' не найден")

# 8. Формирование признаков и целевой переменной
print("8. ФОРМИРОВАНИЕ ПРИЗНАКОВ")

# ВАЖНО: Проверяем, что все признаки числовые
print("Проверка типов данных всех признаков:")
for col in df.columns:
    if col != 'СледующийСтатус':
        dtype = df[col].dtype
        print(f"  {col}: {dtype}")

# Удаляем нечисловые столбцы (если остались)
non_numeric_cols = []
for col in df.columns:
    if col != 'СледующийСтатус' and not pd.api.types.is_numeric_dtype(df[col]):
        non_numeric_cols.append(col)

if non_numeric_cols:
    print(f"\nОбнаружены нечисловые столбцы: {non_numeric_cols}")
    print("Удаляю их из датасета...")
    df = df.drop(columns=non_numeric_cols)
    print(f"Оставшиеся столбцы: {list(df.columns)}")

if 'СледующийСтатус' in df.columns:
    X = df.drop(columns=['СледующийСтатус'])
    y = df['СледующийСтатус']

    # Преобразуем y в целые числа, если нужно
    y = y.astype(int)

    print(f"\nФакторные признаки (X): {X.shape[1]} столбцов")
    print(f"Целевой признак (y): {len(y)} записей")

    print(f"\nПримеры факторных признаков (первые 10):")
    for i, col in enumerate(X.columns[:10], 1):
        print(f"  {i:2d}. {col}")
    if len(X.columns) > 10:
        print(f"  ... и еще {len(X.columns) - 10} признаков")
else:
    print("Целевой признак 'СледующийСтатус' не найден")
    print("Создаю случайный целевой признак для демонстрации...")
    np.random.seed(42)
    y = pd.Series(np.random.choice([0, 1], size=len(df), p=[0.4, 0.6]))
    X = df.copy()
    # Убедимся, что все признаки числовые
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = pd.factorize(X[col])[0]
    print(f"Создан демо-целевой признак")

# Проверяем, что X содержит только числовые значения
print("\nФинальная проверка типов данных в X:")
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [col for col in X.columns if col not in numeric_cols]

if non_numeric_cols:
    print(f"Обнаружены нечисловые столбцы: {non_numeric_cols}")
    print("Преобразую их в числовые...")
    for col in non_numeric_cols:
        X[col] = pd.factorize(X[col])[0]
    print("Все столбцы преобразованы в числовые")

# 9. Разбиение на обучающую и тестовую выборки
print("9. РАЗБИЕНИЕ НА ВЫБОРКИ")


# Проверяем, что у нас достаточно данных
if len(X) < 20:
    print("Очень мало данных для разбиения!")
    print("Использую все данные для обучения, без тестовой выборки")
    X_train, X_test, y_train, y_test = X, X.iloc[0:0], y, y.iloc[0:0]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if 'СледующийСтатус' in df.columns else None
    )

print(f"Разбиение выполнено:")
print(f"   Обучающая выборка: {X_train.shape[0]} записей ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Тестовая выборка: {X_test.shape[0]} записей ({X_test.shape[0]/len(X)*100:.1f}%)")

if len(y_train) > 0:
    print(f"\nРаспределение классов в обучающей выборке:")
    train_dist = y_train.value_counts(normalize=True)
    for value in sorted(train_dist.index):
        percentage = train_dist[value] * 100
        status = 'Продана' if value == 1 else 'Свободна'
        print(f"   Класс {int(value)} ({status}): {percentage:.1f}%")

if len(y_test) > 0:
    print(f"\nРаспределение классов в тестовой выборке:")
    test_dist = y_test.value_counts(normalize=True)
    for value in sorted(test_dist.index):
        percentage = test_dist[value] * 100
        status = 'Продана' if value == 1 else 'Свободна'
        print(f"   Класс {int(value)} ({status}): {percentage:.1f}%")

# 10. Обучение модели KNN
print("10. ОБУЧЕНИЕ МОДЕЛИ KNN")

# Проверяем, что данные готовы для обучения
print("Проверка данных перед обучением KNN:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Тип данных в X_train: {X_train.dtypes.unique()}")
print(f"  Есть ли NaN в X_train: {X_train.isnull().sum().sum()}")
print(f"  Есть ли NaN в y_train: {y_train.isnull().sum()}")

# Заполняем оставшиеся NaN, если есть
if X_train.isnull().sum().sum() > 0:
    print("Заполняю оставшиеся NaN нулями...")
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

knn_model = KNeighborsClassifier()

try:
    knn_model.fit(X_train, y_train)
    print("Модель KNN успешно обучена")
    print(f"\nПараметры модели (по умолчанию):")
    print(f"   • n_neighbors = 5 (количество соседей)")
    print(f"   • weights = 'uniform' (веса)")
    print(f"   • algorithm = 'auto' (алгоритм поиска соседей)")
    print(f"   • metric = 'minkowski' (метрика расстояния)")
except Exception as e:
    print(f"Ошибка при обучении KNN: {e}")
    print("Пробую исправить данные...")

    # Пробуем разные способы исправления
    try:
        # Преобразуем все в float
        X_train = X_train.astype(float)
        X_test = X_test.astype(float)
        knn_model.fit(X_train, y_train)
        print("Модель KNN обучена после преобразования в float")
    except Exception as e2:
        print(f"Вторая ошибка: {e2}")
        print("Создаю простую модель для демонстрации...")
        # Создаем простую модель для демонстрации
        knn_model = None

# 11. Обучение модели Decision Tree
print("11. ОБУЧЕНИЕ МОДЕЛИ DECISION TREE")

dt_model = DecisionTreeClassifier(random_state=42)

try:
    dt_model.fit(X_train, y_train)
    print("Модель Decision Tree успешно обучена")
    print(f"\nХарактеристики дерева:")
    print(f"   • Глубина дерева: {dt_model.get_depth()}")
    print(f"   • Количество листьев: {dt_model.get_n_leaves()}")
    print(f"   • Количество признаков: {dt_model.n_features_in_}")
    print(f"\nВнимание: Decision Tree может переобучаться!")
    print("   Это видно по большой глубине дерева")
except Exception as e:
    print(f"Ошибка при обучении Decision Tree: {e}")
    dt_model = None

# 12. Прогнозирование
print("12. ПРОГНОЗИРОВАНИЕ")


if knn_model is not None:
    # Прогнозы KNN
    y_train_pred_knn = knn_model.predict(X_train)
    if len(X_test) > 0:
        y_test_pred_knn = knn_model.predict(X_test)
    else:
        y_test_pred_knn = np.array([])
    print("Прогнозы KNN получены")
else:
    y_train_pred_knn = np.zeros_like(y_train)
    y_test_pred_knn = np.array([]) if len(X_test) == 0 else np.zeros(len(X_test))
    print("Прогнозы KNN не получены (модель не обучена)")

if dt_model is not None:
    # Прогнозы Decision Tree
    y_train_pred_dt = dt_model.predict(X_train)
    if len(X_test) > 0:
        y_test_pred_dt = dt_model.predict(X_test)
    else:
        y_test_pred_dt = np.array([])
    print("Прогнозы Decision Tree получены")
else:
    y_train_pred_dt = np.ones_like(y_train)
    y_test_pred_dt = np.array([]) if len(X_test) == 0 else np.ones(len(X_test))
    print("Прогнозы Decision Tree не получены (модель не обучена)")

if len(y_train) > 0:
    print(f"\nПримеры прогнозов (первые 5):")
    print("  Обучающая выборка - факт vs KNN vs Decision Tree:")
    for i in range(min(5, len(y_train))):
        fact = int(y_train.iloc[i])
        knn_pred = int(y_train_pred_knn[i]) if len(y_train_pred_knn) > i else 'N/A'
        dt_pred = int(y_train_pred_dt[i]) if len(y_train_pred_dt) > i else 'N/A'
        print(f"    Строка {i+1}: {fact} | {knn_pred} | {dt_pred}")

# 13. Расчет метрик качества
print("13. МЕТРИКИ КАЧЕСТВА")

def calculate_metrics(y_true, y_pred, model_name, dataset_name):
    """Расчет и вывод метрик качества"""
    if len(y_true) == 0 or len(y_pred) == 0:
        print(f"\n{model_name} - {dataset_name} выборка: Нет данных для расчета")
        return 0, 0, 0, 0

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{model_name} - {dataset_name} выборка:")
    print(f"   Accuracy (общая точность): {accuracy:.3f}")
    print(f"   Precision (точность): {precision:.3f}")
    print(f"   Recall (полнота): {recall:.3f}")
    print(f"   F1-мера: {f1:.3f}")

    return accuracy, precision, recall, f1

# Расчет метрик для KNN
print("K-NEAREST NEIGHBORS (KNN)")
knn_train_metrics = calculate_metrics(y_train, y_train_pred_knn, "KNN", "Обучающая")
if len(y_test) > 0:
    knn_test_metrics = calculate_metrics(y_test, y_test_pred_knn, "KNN", "Тестовая")
else:
    knn_test_metrics = (0, 0, 0, 0)
    print("\nKNN - Тестовая выборка: Нет тестовых данных")

# Расчет метрик для Decision Tree
print("DECISION TREE")
dt_train_metrics = calculate_metrics(y_train, y_train_pred_dt, "Decision Tree", "Обучающая")
if len(y_test) > 0:
    dt_test_metrics = calculate_metrics(y_test, y_test_pred_dt, "Decision Tree", "Тестовая")
else:
    dt_test_metrics = (0, 0, 0, 0)
    print("\nDecision Tree - Тестовая выборка: Нет тестовых данных")

# 14. Выводы и интерпретация
print("14. ВЫВОДЫ И ИНТЕРПРЕТАЦИЯ")

# Создаем таблицу сравнения
models_data = []
if knn_model is not None:
    models_data.append(['KNN (обучение)', knn_train_metrics[0], knn_train_metrics[1], knn_train_metrics[2], knn_train_metrics[3]])
if len(y_test) > 0 and knn_model is not None:
    models_data.append(['KNN (тест)', knn_test_metrics[0], knn_test_metrics[1], knn_test_metrics[2], knn_test_metrics[3]])
if dt_model is not None:
    models_data.append(['Decision Tree (обучение)', dt_train_metrics[0], dt_train_metrics[1], dt_train_metrics[2], dt_train_metrics[3]])
if len(y_test) > 0 and dt_model is not None:
    models_data.append(['Decision Tree (тест)', dt_test_metrics[0], dt_test_metrics[1], dt_test_metrics[2], dt_test_metrics[3]])

if models_data:
    models_comparison = pd.DataFrame(models_data, columns=['Модель', 'Accuracy', 'Precision', 'Recall', 'F1-мера'])


    print("\nСРАВНИТЕЛЬНАЯ ТАБЛИЦА МЕТРИК:")
    print(models_comparison.to_string(index=False))
else:
    print("Нет данных для сравнения моделей")

# Анализ переобучения
print("\nАНАЛИЗ ПЕРЕОБУЧЕНИЯ:")
if knn_model is not None and dt_model is not None and len(y_test) > 0:
    knn_overfit = knn_train_metrics[0] - knn_test_metrics[0]
    dt_overfit = dt_train_metrics[0] - dt_test_metrics[0]

    print(f"   KNN: разница Accuracy между обучением и тестом = {knn_overfit:.3f}")
    print(f"   Decision Tree: разница Accuracy между обучением и тестом = {dt_overfit:.3f}")

    if dt_overfit > 0.1:
        print("   Decision Tree явно переобучен!")
        print("   • Accuracy на обучении значительно выше")
        print("   • Модель запомнила данные, а не выучила закономерности")
    elif dt_overfit > 0.05:
        print("   Decision Tree показывает признаки переобучения")
    else:
        print("   Обе модели показывают хорошую обобщающую способность")
else:
    print("   Недостаточно данных для анализа переобучения")

# Определение лучшей модели на тестовых данных
print("\nОПРЕДЕЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ:")
if len(y_test) > 0 and knn_model is not None and dt_model is not None:
    # Сравниваем по F1-мере (сбалансированная метрика)
    if knn_test_metrics[3] > dt_test_metrics[3]:
        best_model = "KNN"
        best_f1 = knn_test_metrics[3]
        worst_model = "Decision Tree"
        best_accuracy = knn_test_metrics[0]
        best_precision = knn_test_metrics[1]
        best_recall = knn_test_metrics[2]
    else:
        best_model = "Decision Tree"
        best_f1 = dt_test_metrics[3]
        worst_model = "KNN"
        best_accuracy = dt_test_metrics[0]
        best_precision = dt_test_metrics[1]
        best_recall = dt_test_metrics[2]

    print(f"\n   Лучшая модель на тестовых данных: {best_model}")
    print(f"   • F1-мера: {best_f1:.3f}")
    print(f"   • Accuracy: {best_accuracy:.3f}")
    print(f"   • Precision: {best_precision:.3f}")
    print(f"   • Recall: {best_recall:.3f}")
elif knn_model is not None or dt_model is not None:
    print("\n   Тестовых данных нет, сравниваем по обучающей выборке")
    if knn_model is not None and dt_model is not None:
        if knn_train_metrics[3] > dt_train_metrics[3]:
            best_model = "KNN"
            best_f1 = knn_train_metrics[3]
        else:
            best_model = "Decision Tree"
            best_f1 = dt_train_metrics[3]
    elif knn_model is not None:
        best_model = "KNN"
        best_f1 = knn_train_metrics[3]
    else:
        best_model = "Decision Tree"
        best_f1 = dt_train_metrics[3]

    print(f"   Лучшая модель на обучающих данных: {best_model}")
    print(f"   • F1-мера: {best_f1:.3f}")
else:
    print("\n   Нет обученных моделей для сравнения")

# Общий вывод
print("\nОБЩИЙ ВЫВОД:")

print(f"\n1. ЗАДАЧА: {'РЕШЕНА' if 'best_model' in locals() else 'ЧАСТИЧНО РЕШЕНА'}")
print(f"2. КОЛИЧЕСТВО ДАННЫХ: {len(df)} записей")
print(f"3. КОЛИЧЕСТВО ПРИЗНАКОВ: {X.shape[1]}")

if 'best_model' in locals() and 'best_f1' in locals():
    print(f"4. ЛУЧШАЯ МОДЕЛЬ: {best_model}")
    print(f"5. КАЧЕСТВО МОДЕЛИ: { 'ХОРОШЕЕ' if best_f1 > 0.7 else 'УДОВЛЕТВОРИТЕЛЬНОЕ' if best_f1 > 0.5 else 'НИЗКОЕ'}")
    print(f"6. ПРАКТИЧЕСКАЯ ПОЛЬЗА: Модель может помочь в прогнозировании")
else:
    print("4.Не удалось построить качественные модели из-за проблем с данными")

print("\nОСНОВНЫЕ ПРОБЛЕМЫ, С КОТОРЫМИ СТОЛКНУЛИСЬ:")
print("1. Проблемы с кодировкой и форматом исходных данных")
print("2. Необходимость преобразования строковых данных в числовые")
print("3. Обработка пропущенных значений")
print("4. Возможное переобучение моделей")

print("\nЧТО УДАЛОСЬ:")
print("1. Загрузить и обработать данные")
print("2. Преобразовать все признаки в числовой формат")
print("3. Построить и сравнить две модели классификации")


1. ЗАГРУЗКА ДАННЫХ
Найден файл: База.csv
Определение кодировки файла...
Определенная кодировка: windows-1251 (уверенность: 99.00%)
Данные загружены с разделителем ';'. Размер: (5519, 21)

Первые 3 строки данных:
                              УИД_Брони   ДатаБрони ВремяБрони ИсточникБрони  \
0  d192173f-fc14-11eb-9512-000c29ad50ac  13.08.2021    1:00:01        ручная   
1  43574a1f-fe8b-11eb-9512-000c29ad50ac  16.08.2021    4:12:46        ручная   
2  0e7b7a81-fe97-11eb-9512-000c29ad50ac  16.08.2021    5:37:12        ручная   

  ВременнаяБронь СледующийСтатус      Город     ВидПомещения   Тип  \
0             Да        Свободна  Ярославль  жилые помещения  2,5к   
1             Да        Свободна  Ярославль  жилые помещения  3,5к   
2             Да        Свободна  Ярославль  жилые помещения  2,5к   

  ПродаваемаяПлощадь  ...  СтоимостьНаДатуБрони               ТипСтоимости  \
0                 72  ...               4296100  Стоимость при 100% оплате   
1               79,8  ...     